In [2]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm

pd.set_option("display.max_columns", 200)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] 
DATA_DIR = PROJECT_ROOT / "data"

In [6]:
BRONZE_ROOT = Path("data/bronze/weather/season=2024")
SILVER_ROOT = Path("data/silver/fact_weather")

print("bronze root exists:", BRONZE_ROOT.exists())
SILVER_ROOT.mkdir(parents=True, exist_ok=True)


bronze root exists: False


In [6]:
test_path = BRONZE_ROOT / "round=1" / "session=R" / "data.parquet"
df1 = pd.read_parquet(test_path, engine="pyarrow")

df1.head()


,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,season,event_round,session_type,ingestion_ts
0,0 days 00:00:14.093000,18.9,46.0,1017.1,False,26.5,162,0.9,2024,1,R,2025-12-15 23:39:57.830341
1,0 days 00:01:14.084000,18.9,46.0,1017.0,False,26.5,55,1.0,2024,1,R,2025-12-15 23:39:57.830341
2,0 days 00:02:14.093000,18.9,46.0,1017.0,False,26.5,55,1.0,2024,1,R,2025-12-15 23:39:57.830341
3,0 days 00:03:14.090000,18.9,45.0,1017.0,False,26.2,85,1.1,2024,1,R,2025-12-15 23:39:57.830341
4,0 days 00:04:14.091000,18.9,46.0,1017.0,False,26.2,178,1.0,2024,1,R,2025-12-15 23:39:57.830341


In [7]:
df1.dtypes


Time             timedelta64[ns]
AirTemp                  float64
Humidity                 float64
Pressure                 float64
Rainfall                    bool
TrackTemp                float64
WindDirection              int64
WindSpeed                float64
season                     int64
event_round                int64
session_type              object
ingestion_ts      datetime64[us]
dtype: object

In [8]:
def clean_weather_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Convert Time (timedelta) → seconds
    df["Time_seconds"] = df["Time"].dt.total_seconds()


    # 3. Drop raw Time column
    df = df.drop(columns=["Time"])

    return df



In [9]:
df_clean = clean_weather_df(df1)

df_clean.head()


,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,season,event_round,session_type,ingestion_ts,Time_seconds
0,18.9,46.0,1017.1,False,26.5,162,0.9,2024,1,R,2025-12-15 23:39:57.830341,14.093
1,18.9,46.0,1017.0,False,26.5,55,1.0,2024,1,R,2025-12-15 23:39:57.830341,74.084
2,18.9,46.0,1017.0,False,26.5,55,1.0,2024,1,R,2025-12-15 23:39:57.830341,134.093
3,18.9,45.0,1017.0,False,26.2,85,1.1,2024,1,R,2025-12-15 23:39:57.830341,194.090
4,18.9,46.0,1017.0,False,26.2,178,1.0,2024,1,R,2025-12-15 23:39:57.830341,254.091


In [10]:
df_clean.dtypes


AirTemp                 float64
Humidity                float64
Pressure                float64
Rainfall                   bool
TrackTemp               float64
WindDirection             int64
WindSpeed               float64
season                    int64
event_round               int64
session_type             object
ingestion_ts     datetime64[us]
Time_seconds            float64
dtype: object

In [11]:
files = list(BRONZE_ROOT.rglob("data.parquet"))
print(f"Found {len(files)} Bronze weather files")

for src in tqdm(files, desc="Writing Silver weather"):
    df = pd.read_parquet(src, engine="pyarrow")
    df_clean = clean_weather_df(df)

    # Preserve partition structure
    relative_path = src.relative_to(BRONZE_ROOT)
    target_path = SILVER_ROOT / relative_path

    target_path.parent.mkdir(parents=True, exist_ok=True)
    df_clean.to_parquet(target_path, index=False, engine="pyarrow")


Found 24 Bronze weather files


Writing Silver weather: 100%|██████████| 24/24 [00:01<00:00, 17.26it/s]


In [12]:
sample = Path("data/silver/fact_weather/round=24/session=R/data.parquet")
df_final = pd.read_parquet(sample, engine="pyarrow")

df_final.head()


,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,season,event_round,session_type,ingestion_ts,Time_seconds
0,27.8,42.0,1017.3,False,37.2,292,1.0,2024,24,R,2025-12-15 23:50:20.007146,56.823
1,27.8,43.0,1017.3,False,37.1,304,2.7,2024,24,R,2025-12-15 23:50:20.007146,116.821
2,27.7,43.0,1017.2,False,36.8,324,2.5,2024,24,R,2025-12-15 23:50:20.007146,176.825
3,27.7,43.0,1017.2,False,36.8,277,2.0,2024,24,R,2025-12-15 23:50:20.007146,236.841
4,27.6,44.0,1017.2,False,36.7,308,1.7,2024,24,R,2025-12-15 23:50:20.007146,296.854
